# Validation Feature Skew Revisit Sprint

This notebook starts from the current public-winning replacement path and revisits skew handling in a narrow way: XGBoost keeps the lean replacement baseline, and we add only a few focused log-transform candidates for the most plausibly useful heavy-tailed numeric features.

Goals:
- preserve the current public-winning replacement path as the baseline
- test a very small XGBoost-only skew-log family instead of the earlier broad skew bundle
- keep CatBoost fixed on the richer additive ordinal baseline
- export versioned submissions from the strict winner only

## 1. Validation Protocol

Workflow:
1. Build the cleaned dataset and the current public-winning replacement baseline.
2. Create a few XGBoost-only skew-log paths on top of that winner, focused on the most plausible heavy-tailed continuous features.
3. Scout the skew variants with the same bag5 XGBoost/CatBoost blend.
4. Re-run strict validation on the same small skew candidate set.
5. Re-check single-seed, bag5, and bag7 on the winning skew path before exporting files.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

try:
    from catboost import CatBoostRegressor
    catboost_import_error = None
except Exception as error:
    CatBoostRegressor = None
    catboost_import_error = error

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
target = "SalePrice"
y_full = train_df[target]

def log_rmse(y_true, y_pred):
    clipped = np.clip(y_pred, a_min=1, a_max=None)
    return root_mean_squared_error(np.log(y_true), np.log(clipped))

def filter_known_outliers(df, target_series):
    mask = pd.Series(True, index=df.index)
    if "GrLivArea" in df.columns:
        mask &= ~((df["GrLivArea"] > 4000) & (target_series < 300000))
    return df.loc[mask].copy(), target_series.loc[mask].copy()

def make_regression_bins(target_series, n_bins=10):
    return pd.qcut(np.log(target_series), q=n_bins, labels=False, duplicates="drop")

In [2]:
X_probe = train_df.drop(columns=[target, "Id"], errors="ignore").copy()
X_probe_clean, _ = filter_known_outliers(X_probe, y_full)

numeric_skew = (
    X_probe_clean.select_dtypes(include="number")
    .apply(lambda column: column.dropna().skew())
    .sort_values(ascending=False)
    .rename("raw_numeric_skew")
    .to_frame()
)

numeric_skew.head(20)

,raw_numeric_skew
MiscVal,24.460085
PoolArea,15.948945
LotArea,12.573925
3SsnPorch,10.297106
LowQualFinSF,9.004955
KitchenAbvGr,4.484883
BsmtFinSF2,4.251925
ScreenPorch,4.118929
BsmtHalfBath,4.100114
EnclosedPorch,3.087164


## 2. Model-Specific Feature Paths

The CatBoost branch keeps the proven additive ordinal baseline. The XGBoost branch is where we test replacement-style representations and selective dropping of source columns.

In [3]:
def add_current_best_features(df):
    df = df.copy()

    def add_sum_feature(new_name, columns):
        existing = [column for column in columns if column in df.columns]
        if existing:
            df[new_name] = df[existing].fillna(0).sum(axis=1)

    add_sum_feature("total_sf", ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"])
    add_sum_feature("total_porch_sf", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"])
    add_sum_feature("total_bathrooms", ["FullBath", "BsmtFullBath"])

    half_bath = df["HalfBath"].fillna(0) if "HalfBath" in df.columns else 0
    bsmt_half_bath = df["BsmtHalfBath"].fillna(0) if "BsmtHalfBath" in df.columns else 0
    if "total_bathrooms" in df.columns or isinstance(half_bath, pd.Series) or isinstance(bsmt_half_bath, pd.Series):
        base_total_bathrooms = df["total_bathrooms"] if "total_bathrooms" in df.columns else 0
        df["total_bathrooms"] = base_total_bathrooms + 0.5 * (half_bath + bsmt_half_bath)

    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["house_age_at_sale"] = df["YrSold"] - df["YearBuilt"]
    if {"YrSold", "YearRemodAdd"}.issubset(df.columns):
        df["years_since_remodel"] = df["YrSold"] - df["YearRemodAdd"]
    if {"YrSold", "GarageYrBlt"}.issubset(df.columns):
        df["garage_age_at_sale"] = df["YrSold"] - df["GarageYrBlt"]
    if {"OverallQual", "OverallCond"}.issubset(df.columns):
        df["quality_condition_score"] = df["OverallQual"] * df["OverallCond"]
    if {"GrLivArea", "TotRmsAbvGrd"}.issubset(df.columns):
        rooms = df["TotRmsAbvGrd"].replace(0, np.nan)
        df["sf_per_room"] = df["GrLivArea"] / rooms
    if "GarageArea" in df.columns:
        df["has_garage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
    if "TotalBsmtSF" in df.columns:
        df["has_basement"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)

    if {"1stFlrSF", "2ndFlrSF", "total_sf"}.issubset(df.columns):
        total_sf = df["total_sf"].replace(0, np.nan)
        df["first_floor_share"] = df["1stFlrSF"].fillna(0) / total_sf
        df["second_floor_share"] = df["2ndFlrSF"].fillna(0) / total_sf
        if "GrLivArea" in df.columns:
            df["above_ground_share"] = df["GrLivArea"].fillna(0) / total_sf

    if {"LotFrontage", "LotArea"}.issubset(df.columns):
        lot_area = df["LotArea"].replace(0, np.nan)
        df["frontage_to_lot_ratio"] = df["LotFrontage"].fillna(0) / lot_area

    if {"OverallQual", "GrLivArea"}.issubset(df.columns):
        df["qual_x_gr_liv_area"] = df["OverallQual"] * df["GrLivArea"]
    if {"OverallQual", "1stFlrSF"}.issubset(df.columns):
        df["qual_x_first_floor_sf"] = df["OverallQual"] * df["1stFlrSF"]

    for column in ["LotFrontage", "MasVnrArea", "GarageYrBlt"]:
        if column in df.columns:
            df[f"{column}_was_missing"] = df[column].isna().astype(int)

    return df


def map_ordinal_feature(series, mapping, missing_token="Missing"):
    return series.fillna(missing_token).astype(str).map(mapping).fillna(0).astype(float)


ordinal_maps = {
    "ExterQual": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "ExterCond": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtQual": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtCond": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "HeatingQC": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "KitchenQual": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "FireplaceQu": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageQual": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "GarageCond": {"Missing": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "PoolQC": {"Missing": 0, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5},
    "BsmtExposure": {"Missing": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"Missing": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"Missing": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "Functional": {"Missing": 0, "Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8},
    "GarageFinish": {"Missing": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Fence": {"Missing": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "LotShape": {"Missing": 0, "IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "LandSlope": {"Missing": 0, "Sev": 1, "Mod": 2, "Gtl": 3},
    "PavedDrive": {"Missing": 0, "N": 0, "P": 1, "Y": 2},
    "Street": {"Missing": 0, "Grvl": 0, "Pave": 1},
    "Alley": {"Missing": 0, "Grvl": 1, "Pave": 2},
    "CentralAir": {"Missing": 0, "N": 0, "Y": 1},
}

ordered_raw_columns = list(ordinal_maps.keys())


def add_quality_ordinal_family(df):
    df = df.copy()

    for column, mapping in ordinal_maps.items():
        if column in df.columns:
            ordinal_column = f"{column.lower()}_ordinal"
            df[ordinal_column] = map_ordinal_feature(df[column], mapping)

    quality_columns = [
        column for column in [
            "exterqual_ordinal",
            "bsmtqual_ordinal",
            "heatingqc_ordinal",
            "kitchenqual_ordinal",
            "fireplacequ_ordinal",
            "garagequal_ordinal",
        ]
        if column in df.columns
    ]
    if quality_columns:
        df["quality_ordinal_sum"] = df[quality_columns].sum(axis=1)

    basement_columns = [
        column for column in [
            "bsmtexposure_ordinal",
            "bsmtfintype1_ordinal",
            "bsmtfintype2_ordinal",
        ]
        if column in df.columns
    ]
    if basement_columns:
        df["basement_finish_score"] = df[basement_columns].sum(axis=1)

    if {"quality_ordinal_sum", "GrLivArea"}.issubset(df.columns):
        df["quality_ordinal_x_liv_area"] = df["quality_ordinal_sum"] * df["GrLivArea"]

    return df


def add_core_size_log_family(df):
    df = df.copy()
    columns = ["LotArea", "LotFrontage", "GrLivArea", "total_sf"]
    for column in columns:
        if column in df.columns:
            values = df[column].fillna(0).clip(lower=0)
            df[f"log1p_{column.lower()}"] = np.log1p(values)
    return df


def add_amenity_area_log_family(df):
    df = df.copy()
    columns = ["MasVnrArea", "GarageArea", "total_porch_sf"]
    for column in columns:
        if column in df.columns:
            values = df[column].fillna(0).clip(lower=0)
            df[f"log1p_{column.lower()}"] = np.log1p(values)
    return df


skew_feature_builders = {
    "core_size_logs": add_core_size_log_family,
    "amenity_area_logs": add_amenity_area_log_family,
}

feature_path_configs = {
    "current_winner_path": {
        "xgb_extra_families": [],
        "xgb_drop_raw_ordered": True,
    },
    "winner_plus_core_size_logs": {
        "xgb_extra_families": ["core_size_logs"],
        "xgb_drop_raw_ordered": True,
    },
    "winner_plus_amenity_area_logs": {
        "xgb_extra_families": ["amenity_area_logs"],
        "xgb_drop_raw_ordered": True,
    },
    "winner_plus_focused_skew_logs": {
        "xgb_extra_families": ["core_size_logs", "amenity_area_logs"],
        "xgb_drop_raw_ordered": True,
    },
}


def describe_feature_path(config):
    labels = []
    if config["xgb_extra_families"]:
        labels.extend(config["xgb_extra_families"])
    if config.get("xgb_drop_raw_ordered"):
        labels.append("drop_raw_ordered")
    return ", ".join(labels) or "cat_baseline_plus_xgb_additive_baseline"


def clean_numeric_issues(df):
    df = df.copy()
    numeric_columns = df.select_dtypes(include="number").columns
    df[numeric_columns] = df[numeric_columns].replace([np.inf, -np.inf], np.nan)
    return df


def build_cat_feature_matrix(df):
    engineered = add_current_best_features(df)
    engineered = add_quality_ordinal_family(engineered)
    return clean_numeric_issues(engineered)


def build_xgb_feature_matrix(df, config):
    engineered = add_current_best_features(df)
    engineered = add_quality_ordinal_family(engineered)

    for family_name in config["xgb_extra_families"]:
        engineered = skew_feature_builders[family_name](engineered)

    if config.get("xgb_drop_raw_ordered"):
        engineered = engineered.drop(columns=ordered_raw_columns, errors="ignore")

    return clean_numeric_issues(engineered)


X_full_raw = train_df.drop(columns=[target, "Id"], errors="ignore").copy()
X_test_raw = test_df.drop(columns=["Id"], errors="ignore").copy()
X_full_clean, y_clean = filter_known_outliers(X_full_raw, y_full)

cat_train_baseline = build_cat_feature_matrix(X_full_clean)
cat_test_baseline = build_cat_feature_matrix(X_test_raw)

xgb_train_matrices = {
    feature_path_name: build_xgb_feature_matrix(X_full_clean, config)
    for feature_path_name, config in feature_path_configs.items()
}
xgb_test_matrices = {
    feature_path_name: build_xgb_feature_matrix(X_test_raw, config)
    for feature_path_name, config in feature_path_configs.items()
}
cat_train_matrices = {
    feature_path_name: cat_train_baseline
    for feature_path_name in feature_path_configs
}
cat_test_matrices = {
    feature_path_name: cat_test_baseline
    for feature_path_name in feature_path_configs
}

feature_path_summary = pd.DataFrame([
    {
        "feature_path_name": feature_path_name,
        "feature_strategy": describe_feature_path(config),
        "n_xgb_features": xgb_train_matrices[feature_path_name].shape[1],
        "n_cat_features": cat_train_matrices[feature_path_name].shape[1],
        "n_xgb_numeric_features": int(xgb_train_matrices[feature_path_name].select_dtypes(include="number").shape[1]),
        "n_xgb_categorical_features": int(xgb_train_matrices[feature_path_name].select_dtypes(exclude="number").shape[1]),
    }
    for feature_path_name, config in feature_path_configs.items()
]).sort_values("n_xgb_features").reset_index(drop=True)

X_full = xgb_train_matrices["current_winner_path"]
X_test_engineered = xgb_test_matrices["current_winner_path"]

feature_path_summary

,feature_path_name,feature_strategy,n_xgb_features,n_cat_features,n_xgb_numeric_features,n_xgb_categorical_features
0,current_winner_path,drop_raw_ordered,101,123,80,21
1,winner_plus_amenity_area_logs,"amenity_area_logs, drop_raw_ordered",104,123,83,21
2,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",105,123,84,21
3,winner_plus_focused_skew_logs,"core_size_logs, amenity_area_logs, drop_raw_or...",108,123,87,21


## 3. Fixed Blend Candidate Set

Keep the model family fixed and let the XGBoost representation vary. CatBoost stays on the stable additive ordinal baseline for every candidate.

In [4]:
baseline_xgb_blend_params = {
    "n_estimators": 900,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 3,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_lambda": 0.5,
}

baseline_cat_blend_params = {
    "iterations": 800,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 3.0,
}

bag5_seed_pairs = [(42, 42), (52, 52), (62, 62), (72, 72), (82, 82)]
bag7_seed_pairs = [(42, 42), (52, 52), (62, 62), (72, 72), (82, 82), (92, 92), (102, 102)]
bag7_weights = [0.30, 0.20, 0.15, 0.12, 0.10, 0.08, 0.05]

def split_feature_types(df):
    numeric_columns = df.select_dtypes(include="number").columns.tolist()
    categorical_columns = df.select_dtypes(exclude="number").columns.tolist()
    return numeric_columns, categorical_columns

def build_tree_pipeline(model, df):
    numeric_columns, categorical_columns = split_feature_types(df)
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                numeric_columns,
            ),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

def prepare_catboost_native_features(df):
    catboost_df = df.copy()
    categorical_columns = catboost_df.select_dtypes(exclude="number").columns.tolist()
    for column in categorical_columns:
        catboost_df[column] = catboost_df[column].fillna("Missing").astype(str)
    return catboost_df, categorical_columns

def fit_predict_xgb(X_train, y_train, X_valid, params, random_seed):
    model = build_tree_pipeline(
        XGBRegressor(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            min_child_weight=params["min_child_weight"],
            subsample=params["subsample"],
            colsample_bytree=params["colsample_bytree"],
            reg_lambda=params["reg_lambda"],
            objective="reg:squarederror",
            random_state=random_seed,
            n_jobs=-1,
            verbosity=0,
        ),
        X_train,
    )
    model.fit(X_train, y_train)
    return model.predict(X_valid)

def fit_predict_cat(X_train, y_train, X_valid, params, random_seed):
    if CatBoostRegressor is None:
        raise RuntimeError(f"CatBoost import failed: {catboost_import_error}")
    X_train_cat, cat_columns = prepare_catboost_native_features(X_train)
    X_valid_cat, _ = prepare_catboost_native_features(X_valid)
    model = CatBoostRegressor(
        iterations=params["iterations"],
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],
        loss_function="RMSE",
        random_seed=random_seed,
        verbose=False,
        cat_features=cat_columns,
    )
    model.fit(X_train_cat, y_train)
    return model.predict(X_valid_cat)

def fit_predict_seed_pair_blend(X_train_xgb, X_train_cat, y_train, X_valid_xgb, X_valid_cat, xgb_seed, cat_seed):
    xgb_preds = fit_predict_xgb(
        X_train_xgb,
        y_train,
        X_valid_xgb,
        params=baseline_xgb_blend_params,
        random_seed=xgb_seed,
    )
    cat_preds = fit_predict_cat(
        X_train_cat,
        y_train,
        X_valid_cat,
        params=baseline_cat_blend_params,
        random_seed=cat_seed,
    )
    return 0.5 * xgb_preds + 0.5 * cat_preds

def average_component_predictions(component_preds, weights=None):
    component_matrix = np.vstack(component_preds)
    if weights is None:
        return component_matrix.mean(axis=0)

    weights_array = np.asarray(weights, dtype=float)
    weights_array = weights_array / weights_array.sum()
    return np.average(component_matrix, axis=0, weights=weights_array)

feature_candidate_configs = {
    feature_path_name: {
        "feature_path_name": feature_path_name,
        "seed_pairs": bag5_seed_pairs,
    }
    for feature_path_name in feature_path_configs
}
feature_candidate_names = list(feature_candidate_configs.keys())

feature_candidate_overview = pd.DataFrame([
    {
        "candidate_name": candidate_name,
        "feature_strategy": describe_feature_path(feature_path_configs[config["feature_path_name"]]),
        "n_seed_pairs": len(config["seed_pairs"]),
    }
    for candidate_name, config in feature_candidate_configs.items()
]).sort_values("candidate_name").reset_index(drop=True)

feature_candidate_overview

,candidate_name,feature_strategy,n_seed_pairs
0,current_winner_path,drop_raw_ordered,5
1,winner_plus_amenity_area_logs,"amenity_area_logs, drop_raw_ordered",5
2,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",5
3,winner_plus_focused_skew_logs,"core_size_logs, amenity_area_logs, drop_raw_or...",5


In [5]:
candidate_snapshot = feature_path_summary.merge(
    feature_candidate_overview[["candidate_name", "n_seed_pairs"]],
    left_on="feature_path_name",
    right_on="candidate_name",
    how="left",
).drop(columns=["candidate_name"])

candidate_snapshot

,feature_path_name,feature_strategy,n_xgb_features,n_cat_features,n_xgb_numeric_features,n_xgb_categorical_features,n_seed_pairs
0,current_winner_path,drop_raw_ordered,101,123,80,21,5
1,winner_plus_amenity_area_logs,"amenity_area_logs, drop_raw_ordered",104,123,83,21,5
2,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",105,123,84,21,5
3,winner_plus_focused_skew_logs,"core_size_logs, amenity_area_logs, drop_raw_or...",108,123,87,21,5


## 3b. Candidate Snapshot

This preview shows how each focused skew path changes the XGBoost representation while CatBoost stays fixed on the richer additive baseline.

## 4. Evaluation Helpers

This section defines outer-validation helpers for model-specific feature paths. Round 2 still ranks candidates by mean outer log RMSE first and uses split-win count only as a secondary tiebreak.

In [6]:
outer_test_size = 0.2
scout_outer_split_seeds = [11, 23, 42]
strict_outer_split_seeds = [11, 23, 42, 67, 101]
scout_top_k = 4

def summarize_outer_results(outer_results):
    selected_by_outer_split = (
        outer_results
        .sort_values(["outer_seed", "outer_validation_log_rmse"])
        .groupby("outer_seed", as_index=False)
        .first()
    )

    selection_counts = (
        selected_by_outer_split["candidate_name"]
        .value_counts()
        .rename_axis("candidate_name")
        .reset_index(name="outer_selection_count")
    )

    summary = (
        outer_results
        .groupby("candidate_name")
        .agg(
            mean_outer_validation_log_rmse=("outer_validation_log_rmse", "mean"),
            std_outer_validation_log_rmse=("outer_validation_log_rmse", "std"),
            min_outer_validation_log_rmse=("outer_validation_log_rmse", "min"),
            max_outer_validation_log_rmse=("outer_validation_log_rmse", "max"),
            n_outer_splits=("outer_validation_log_rmse", "size"),
        )
        .reset_index()
        .merge(selection_counts, on="candidate_name", how="left")
        .fillna({"outer_selection_count": 0})
        .sort_values(["mean_outer_validation_log_rmse", "outer_selection_count"], ascending=[True, False])
    )
    summary["outer_selection_count"] = summary["outer_selection_count"].astype(int)
    return selected_by_outer_split, summary

def evaluate_candidates(candidate_config_dict, split_seeds):
    outer_rows = []
    row_index = y_clean.index
    stratify_bins = make_regression_bins(y_clean, n_bins=10)
    all_feature_seed_pairs = sorted({
        (config["feature_path_name"], seed_pair)
        for config in candidate_config_dict.values()
        for seed_pair in config["seed_pairs"]
    })

    for outer_seed in split_seeds:
        train_idx, valid_idx = train_test_split(
            row_index,
            test_size=outer_test_size,
            random_state=outer_seed,
            stratify=stratify_bins.loc[row_index],
        )
        y_train_outer = y_clean.loc[train_idx]
        y_valid_outer = y_clean.loc[valid_idx]

        seed_pair_pred_cache = {}
        for feature_path_name, seed_pair in all_feature_seed_pairs:
            xgb_seed, cat_seed = seed_pair
            seed_pair_pred_cache[(feature_path_name, seed_pair)] = fit_predict_seed_pair_blend(
                xgb_train_matrices[feature_path_name].loc[train_idx],
                cat_train_matrices[feature_path_name].loc[train_idx],
                y_train_outer,
                xgb_train_matrices[feature_path_name].loc[valid_idx],
                cat_train_matrices[feature_path_name].loc[valid_idx],
                xgb_seed=xgb_seed,
                cat_seed=cat_seed,
            )

        for candidate_name, config in candidate_config_dict.items():
            component_preds = [
                seed_pair_pred_cache[(config["feature_path_name"], seed_pair)]
                for seed_pair in config["seed_pairs"]
            ]
            outer_preds = average_component_predictions(component_preds, config.get("weights"))
            outer_rows.append({
                "outer_seed": outer_seed,
                "candidate_name": candidate_name,
                "outer_validation_log_rmse": log_rmse(y_valid_outer, outer_preds),
            })

    outer_results = pd.DataFrame(outer_rows)
    selected_by_outer_split, summary = summarize_outer_results(outer_results)
    return outer_results, selected_by_outer_split, summary

def build_full_candidate_predictions(candidate_name, candidate_config_dict):
    config = candidate_config_dict[candidate_name]
    feature_path_name = config["feature_path_name"]
    component_preds = []

    for xgb_seed, cat_seed in config["seed_pairs"]:
        component_preds.append(
            fit_predict_seed_pair_blend(
                xgb_train_matrices[feature_path_name],
                cat_train_matrices[feature_path_name],
                y_clean,
                xgb_test_matrices[feature_path_name],
                cat_test_matrices[feature_path_name],
                xgb_seed=xgb_seed,
                cat_seed=cat_seed,
            )
        )

    return average_component_predictions(component_preds, config.get("weights"))

## 5. Feature Path Validation

Stage A scouts the focused skew variants quickly; Stage B re-runs strict validation on the same small XGBoost-only skew candidate set while keeping the CatBoost baseline fixed.

In [7]:
anchor_feature_candidates = set(feature_candidate_configs)

feature_scout_results, feature_scout_selected_by_outer_split, feature_scout_summary = evaluate_candidates(
    feature_candidate_configs,
    scout_outer_split_seeds,
)

feature_scout_shortlist = (
    feature_scout_summary
    .sort_values(["mean_outer_validation_log_rmse", "outer_selection_count"], ascending=[True, False])
    .head(scout_top_k)["candidate_name"]
    .tolist()
)
feature_strict_shortlist = sorted(set(feature_scout_shortlist).union(anchor_feature_candidates))
feature_strict_candidate_configs = {
    candidate_name: feature_candidate_configs[candidate_name]
    for candidate_name in feature_strict_shortlist
}

feature_strict_results, feature_strict_selected_by_outer_split, feature_strict_summary = evaluate_candidates(
    feature_strict_candidate_configs,
    strict_outer_split_seeds,
)

best_feature_path_name = feature_strict_summary.iloc[0]["candidate_name"]
best_feature_path_row = feature_strict_summary.loc[
    feature_strict_summary["candidate_name"] == best_feature_path_name
].iloc[0]
feature_path_decision = pd.DataFrame([
    {
        "best_feature_path_name": best_feature_path_name,
        "feature_strategy": describe_feature_path(feature_path_configs[best_feature_path_name]),
        "mean_outer_validation_log_rmse": float(best_feature_path_row["mean_outer_validation_log_rmse"]),
        "std_outer_validation_log_rmse": float(best_feature_path_row["std_outer_validation_log_rmse"]),
        "outer_selection_count": int(best_feature_path_row["outer_selection_count"]),
        "strict_shortlist": ", ".join(feature_strict_shortlist),
    }
])

feature_path_decision

,best_feature_path_name,feature_strategy,mean_outer_validation_log_rmse,std_outer_validation_log_rmse,outer_selection_count,strict_shortlist
0,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",0.106996,0.006113,4,"current_winner_path, winner_plus_amenity_area_..."


## 6. Final Model Check on Winning Feature Path

Before exporting, re-check the usual bagging variants on the winning skew path. Mean outer log RMSE remains the primary ranking signal.

In [8]:
model_candidate_configs = {
    "single_seed": {
        "feature_path_name": best_feature_path_name,
        "seed_pairs": [(42, 42)],
    },
    "bag5_equal": {
        "feature_path_name": best_feature_path_name,
        "seed_pairs": bag5_seed_pairs,
    },
    "bag7_tapered": {
        "feature_path_name": best_feature_path_name,
        "seed_pairs": bag7_seed_pairs,
        "weights": bag7_weights,
    },
}

model_results, model_selected_by_outer_split, model_summary = evaluate_candidates(
    model_candidate_configs,
    strict_outer_split_seeds,
)

best_final_candidate_name = model_summary.iloc[0]["candidate_name"]
best_final_candidate_row = model_summary.loc[
    model_summary["candidate_name"] == best_final_candidate_name
].iloc[0]
test_preds = build_full_candidate_predictions(best_final_candidate_name, model_candidate_configs)

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_preds,
})
submission_tag = "feature_skew_revisit_sprint"
submission_path = f"../submissions/submission_{best_feature_path_name}_{best_final_candidate_name}_{submission_tag}.csv"
submission.to_csv(submission_path, index=False)

selection_summary = pd.DataFrame([
    {
        "best_feature_path_name": best_feature_path_name,
        "winning_feature_strategy": describe_feature_path(feature_path_configs[best_feature_path_name]),
        "best_final_candidate_name": best_final_candidate_name,
        "mean_outer_validation_log_rmse": float(best_final_candidate_row["mean_outer_validation_log_rmse"]),
        "std_outer_validation_log_rmse": float(best_final_candidate_row["std_outer_validation_log_rmse"]),
        "outer_selection_count": int(best_final_candidate_row["outer_selection_count"]),
        "submission_path": submission_path,
    }
])
selection_summary

,best_feature_path_name,winning_feature_strategy,best_final_candidate_name,mean_outer_validation_log_rmse,std_outer_validation_log_rmse,outer_selection_count,submission_path
0,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",bag5_equal,0.106996,0.006113,2,../submissions/submission_winner_plus_core_siz...


In [9]:
submission_path

'../submissions/submission_winner_plus_core_size_logs_bag5_equal_feature_skew_revisit_sprint.csv'

## 7. Diagnostics and Exports

This section reports scout and strict summaries, exports the model shortlist on the winning skew path, and shows the delta versus the current public-winning replacement baseline.

In [10]:
feature_diagnostic_columns = [
    "candidate_name",
    "feature_strategy",
    "n_xgb_features",
    "n_cat_features",
    "mean_outer_validation_log_rmse",
    "std_outer_validation_log_rmse",
    "outer_selection_count",
]

feature_diagnostics = feature_path_summary.rename(columns={"feature_path_name": "candidate_name"})

print("Feature-path scout summary (sorted by mean outer log RMSE):")
display(
    feature_scout_summary
    .merge(feature_diagnostics, on="candidate_name", how="left")
    [feature_diagnostic_columns]
    .sort_values(["mean_outer_validation_log_rmse", "outer_selection_count"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"\nFeature-path strict shortlist ({len(feature_strict_shortlist)} candidates): {feature_strict_shortlist}")
display(
    feature_strict_summary
    .merge(feature_diagnostics, on="candidate_name", how="left")
    [feature_diagnostic_columns]
    .sort_values(["mean_outer_validation_log_rmse", "outer_selection_count"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"\nFinal model check on winning path: {best_feature_path_name}")
display(
    model_summary[[
        "candidate_name",
        "mean_outer_validation_log_rmse",
        "std_outer_validation_log_rmse",
        "outer_selection_count",
    ]]
    .sort_values(["mean_outer_validation_log_rmse", "outer_selection_count"], ascending=[True, False])
    .reset_index(drop=True)
)

Feature-path scout summary (sorted by mean outer log RMSE):


,candidate_name,feature_strategy,n_xgb_features,n_cat_features,mean_outer_validation_log_rmse,std_outer_validation_log_rmse,outer_selection_count
0,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",105,123,0.105078,0.006650,2
1,winner_plus_focused_skew_logs,"core_size_logs, amenity_area_logs, drop_raw_or...",108,123,0.105159,0.006636,1
2,current_winner_path,drop_raw_ordered,101,123,0.105167,0.006538,0
3,winner_plus_amenity_area_logs,"amenity_area_logs, drop_raw_ordered",104,123,0.105338,0.006639,0



Feature-path strict shortlist (4 candidates): ['current_winner_path', 'winner_plus_amenity_area_logs', 'winner_plus_core_size_logs', 'winner_plus_focused_skew_logs']


,candidate_name,feature_strategy,n_xgb_features,n_cat_features,mean_outer_validation_log_rmse,std_outer_validation_log_rmse,outer_selection_count
0,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",105,123,0.106996,0.006113,4
1,winner_plus_focused_skew_logs,"core_size_logs, amenity_area_logs, drop_raw_or...",108,123,0.107068,0.006116,1
2,current_winner_path,drop_raw_ordered,101,123,0.107137,0.006066,0
3,winner_plus_amenity_area_logs,"amenity_area_logs, drop_raw_ordered",104,123,0.107220,0.006111,0



Final model check on winning path: winner_plus_core_size_logs


,candidate_name,mean_outer_validation_log_rmse,std_outer_validation_log_rmse,outer_selection_count
0,bag5_equal,0.106996,0.006113,2
1,bag7_tapered,0.107008,0.006010,2
2,single_seed,0.107445,0.005960,1


In [11]:
strict_component_submission_paths = {}
for candidate_name in model_candidate_configs:
    candidate_preds = build_full_candidate_predictions(candidate_name, model_candidate_configs)
    candidate_submission = pd.DataFrame({
        "Id": test_df["Id"],
        "SalePrice": candidate_preds,
    })
    candidate_path = f"../submissions/submission_{best_feature_path_name}_{candidate_name}_feature_skew_revisit_component.csv"
    candidate_submission.to_csv(candidate_path, index=False)
    strict_component_submission_paths[candidate_name] = candidate_path

pd.Series(strict_component_submission_paths, name="submission_path")

single_seed     ../submissions/submission_winner_plus_core_siz...
bag5_equal      ../submissions/submission_winner_plus_core_siz...
bag7_tapered    ../submissions/submission_winner_plus_core_siz...
Name: submission_path, dtype: object

In [12]:
feature_path_delta_table = (
    feature_strict_results[
        feature_strict_results["candidate_name"].isin(["current_winner_path", best_feature_path_name])
    ]
    .pivot(index="outer_seed", columns="candidate_name", values="outer_validation_log_rmse")
    .reset_index()
)

if best_feature_path_name != "current_winner_path" and "current_winner_path" in feature_path_delta_table.columns:
    feature_path_delta_table["delta_vs_current_winner"] = (
        feature_path_delta_table[best_feature_path_name] - feature_path_delta_table["current_winner_path"]
    )

feature_path_delta_table

candidate_name,outer_seed,current_winner_path,winner_plus_core_size_logs,delta_vs_current_winner
0,11,0.110825,0.110732,-0.000093
1,23,0.098009,0.097752,-0.000257
2,42,0.106667,0.106751,0.000084
3,67,0.106054,0.105780,-0.000274
4,101,0.114129,0.113963,-0.000165


In [13]:
best_single_name = model_summary.iloc[0]["candidate_name"]
best_single_mean = float(model_summary.iloc[0]["mean_outer_validation_log_rmse"])
best_single_std = float(model_summary.iloc[0]["std_outer_validation_log_rmse"])
winning_feature_mean = float(
    feature_strict_summary.loc[
        feature_strict_summary["candidate_name"] == best_feature_path_name,
        "mean_outer_validation_log_rmse",
    ].iloc[0]
)
baseline_current_winner_mean = float(
    feature_strict_summary.loc[
        feature_strict_summary["candidate_name"] == "current_winner_path",
        "mean_outer_validation_log_rmse",
    ].iloc[0]
)

final_submission_decision = pd.DataFrame([
    {
        "recommended_file_to_submit": submission_path,
        "winning_feature_path": best_feature_path_name,
        "winning_feature_strategy": describe_feature_path(feature_path_configs[best_feature_path_name]),
        "winning_model_variant": best_single_name,
        "local_basis": f"Mean-first strict ranking on skew path {best_feature_path_name}",
        "winning_feature_path_mean_outer_validation_log_rmse": winning_feature_mean,
        "baseline_current_winner_mean_outer_validation_log_rmse": baseline_current_winner_mean,
        "feature_path_gain_vs_current_winner": baseline_current_winner_mean - winning_feature_mean,
        "final_model_mean_outer_validation_log_rmse": best_single_mean,
        "final_model_std_outer_validation_log_rmse": best_single_std,
    }
])

final_submission_decision

,recommended_file_to_submit,winning_feature_path,winning_feature_strategy,winning_model_variant,local_basis,winning_feature_path_mean_outer_validation_log_rmse,baseline_current_winner_mean_outer_validation_log_rmse,feature_path_gain_vs_current_winner,final_model_mean_outer_validation_log_rmse,final_model_std_outer_validation_log_rmse
0,../submissions/submission_winner_plus_core_siz...,winner_plus_core_size_logs,"core_size_logs, drop_raw_ordered",bag5_equal,Mean-first strict ranking on skew path winner_...,0.106996,0.107137,0.000141,0.106996,0.006113
